# Frozen USD-major mean-reversion baseline — trades and metrics

This notebook reads the immutable `EXP-0001` artifacts produced by `python -u forex/exploration_4/_run_baseline.py`. It does not rerun, filter, tune, or select the strategy. The frozen verdict is **NO-GO**.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'forex' / 'exploration_4').exists():
        root = candidate
        break
project = root / 'forex' / 'exploration_4'
artifact = project / 'artifacts' / 'runs' / 'EXP-0001'
data_dir = root / 'forex' / 'data' / 'clean'

trades = pd.read_parquet(artifact / 'trades.parquet')
metrics = pd.read_csv(artifact / 'metrics.csv')
costs = pd.read_csv(artifact / 'cost_by_era_hour.csv')
daily = pd.read_csv(artifact / 'daily_marked_R.csv', parse_dates=['date'])
sharpes = pd.read_csv(artifact / 'sharpes.csv')
null_draws = pd.read_csv(artifact / 'random_exit_draws.csv')
dq = pd.read_csv(artifact / 'data_quality_overall.csv')
verdict = json.loads((artifact / 'verdict.json').read_text())

for col in ['signal_time', 'entry_time', 'exit_time']:
    trades[col] = pd.to_datetime(trades[col], utc=True)
display(Markdown(f"**Verdict: {verdict['verdict']}** — {len(trades):,} headline trades; consumed-history base net mean `{verdict['base_net_mean_R']:+.4f} R` with 95% CI `{verdict['base_net_ci'][0]:+.4f}` to `{verdict['base_net_ci'][1]:+.4f}`."))

## Headline, diagnostics, and sealed holdout

In [ ]:
headline = metrics.query("pair == 'POOLED' and scope in ['consumed','holdout']")[['scope','outcome','n','clusters','mean','se','t','ci_low','ci_high','hit_rate']]
display(headline.style.format({'mean':'{:+.4f}','se':'{:.4f}','t':'{:+.2f}','ci_low':'{:+.4f}','ci_high':'{:+.4f}','hit_rate':'{:.1%}'}))
display(sharpes.style.format({'zero_day_sharpe':'{:+.2f}','trade_days_only_sharpe':'{:+.2f}','vol_targeted_sharpe':'{:+.2f}','worst_day_R':'{:+.2f}','max_drawdown_R':'{:.1f}'}))

## Representative trade paths

One consumed-history headline trade is selected per pair with a fixed random seed. These are representative illustrations, not best/worst examples. Entry is the next 5-minute open after the signal; the dashed red line is the frozen compulsory stop; exit is the attained stop fill or the next open after observed reversion.

In [ ]:
sample = (trades.query("era != 'holdout'").groupby('pair', group_keys=False)
          .sample(n=1, random_state=20260808).sort_values('pair'))
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
for ax, (_, t) in zip(axes.flat, sample.iterrows()):
    start = (t.signal_time - pd.Timedelta(minutes=30)).tz_convert(None)
    end = (t.exit_time + pd.Timedelta(minutes=30)).tz_convert(None)
    px = pd.read_parquet(data_dir / f'{t.pair}_1m_clean.parquet', columns=['ts_utc','close'],
                         filters=[('ts_utc','>=',start),('ts_utc','<=',end)])
    px['ts_utc'] = pd.to_datetime(px.ts_utc, utc=True)
    ax.plot(px.ts_utc, px.close, color='0.25', lw=1, label='1m midpoint close')
    ax.scatter(t.entry_time, t.entry_px, marker='^' if t.side > 0 else 'v', s=90, color='#1565c0', zorder=5, label='entry')
    ax.scatter(t.exit_time, t.exit_px, marker='X', s=90, color='#ef6c00', zorder=5, label=f"exit: {t.exit_kind}")
    stop_px = t.entry_px - t.side * t.risk_pips * 0.0001
    ax.axhline(stop_px, color='#c62828', ls='--', lw=1.2, label='frozen stop')
    ax.axvline(t.signal_time, color='#6a1b9a', ls=':', lw=1, label='signal close')
    ax.set_title(f"{t.pair} | {t.side:+.0f} side | {t.gross_R:+.2f}R gross / {t.net_base_vol_R:+.2f}R net")
    ax.set_ylabel('midpoint price')
    ax.legend(fontsize=8, loc='best')
plt.show()

## Trade outcomes and holding time

In [ ]:
trades['holding_minutes'] = (trades.exit_time - trades.entry_time).dt.total_seconds() / 60
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
sns.boxenplot(data=trades, x='pair', y='net_base_vol_R', hue='era', showfliers=False, ax=axes[0])
axes[0].axhline(0, color='black', lw=1)
axes[0].set_title('Base-cost net R by pair and era')
sns.boxplot(data=trades, x='exit_kind', y='holding_minutes', hue='pair', showfliers=False, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Holding time by realized exit type (log scale)')
plt.show()
exit_summary = trades.groupby(['era','exit_kind']).agg(trades=('trade_id','size'), mean_gross_R=('gross_R','mean'), mean_net_R=('net_base_vol_R','mean'), median_hold_min=('holding_minutes','median')).reset_index()
display(exit_summary.style.format({'mean_gross_R':'{:+.3f}','mean_net_R':'{:+.3f}','median_hold_min':'{:.0f}'}))

## Marked daily P&L and drawdown

In [ ]:
portfolio = daily.groupby('date').net_R.sum().sort_index()
equity = portfolio.cumsum()
drawdown = equity - equity.cummax()
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, constrained_layout=True)
axes[0].plot(equity.index, equity, color='#1565c0')
axes[0].axvline(pd.Timestamp('2024-01-01'), color='#ef6c00', ls='--', label='sealed holdout begins')
axes[0].set_title('Cumulative marked base-cost net R (sum across pairs)')
axes[0].legend()
axes[1].fill_between(drawdown.index, drawdown.values, 0, color='#c62828', alpha=.55)
axes[1].set_title('Drawdown in portfolio R units')
plt.show()

## Clock-location and cost economics

In [ ]:
base = costs.query("scenario == 'base' and vol_term == 'vol' and era != 'holdout'")
hour = base.groupby('utc_hour').apply(lambda g: pd.Series({
    'signals': g.signals.sum(),
    'gross_pips': np.average(g.gross_pips, weights=g.signals),
    'cost_pips': np.average(g.cost_pips, weights=g.signals),
    'net_pips': np.average(g.net_pips, weights=g.signals)}), include_groups=False).reset_index()
fig, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(hour.utc_hour, hour.gross_pips, marker='o', label='gross pips')
ax1.plot(hour.utc_hour, hour.cost_pips, marker='o', label='modeled base cost')
ax1.plot(hour.utc_hour, hour.net_pips, marker='o', label='net pips')
ax1.axhline(0, color='black', lw=1)
ax1.axvspan(20.5, 21.5, color='#ef6c00', alpha=.15, label='~21 UTC rollover')
ax1.set_xticks(range(24)); ax1.set_xlabel('entry UTC hour'); ax1.set_ylabel('mean pips / signal')
ax2 = ax1.twinx(); ax2.bar(hour.utc_hour, hour.signals, alpha=.12, color='grey'); ax2.set_ylabel('signals')
ax1.set_title('Consumed-history gross, modeled cost, and net by UTC hour')
ax1.legend(ncol=4, loc='upper center')
plt.show()

## Matched-rate random-exit null

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(null_draws.mean_gross_R, bins=35, stat='density', color='#90caf9', ax=ax)
ax.axvline(verdict['random_exit']['observed_mean_gross_R'], color='#c62828', lw=2, label='observed reversion exit')
ax.axvline(verdict['random_exit']['null_q95'], color='black', ls='--', label='null 95th percentile')
ax.set_title('1,000 exit-duration re-pairing draws')
ax.set_xlabel('mean gross R / signal'); ax.legend(); plt.show()
display(pd.DataFrame([verdict['random_exit']]).drop(columns=['preserves','destroys']).style.format('{:.5f}'))
display(Markdown('The observed exit beats this matched-rate null, but remains **negative**. This isolated exit-timing improvement cannot rescue the failed entry and cost tests.'))

## Data quality and interpretation boundary

In [ ]:
display(dq)
display(Markdown('The source has midpoint OHLC only: `volume` is absent and bid/ask is unavailable. Empty weekend/rollover bins are not treated as tradable. NZDUSD has material late-era 18:00–19:00 UTC holes; see `reports/DATA_QUALITY.md` for per-era/hour feature and path exclusions. Each of the other three pairs is independently negative, so this caveat does not alter the NO-GO verdict.'))